# 06. CAL01 정비 감지·대기·완료와 온도 변화

**담당:** 정수진 · **작성자 표기:** jinny · **Issue:** #75 (상위 #65 → #38)  
**대상 설비:** P1-CR1-CAL01 · **수정일:** 2026-10-16  
**브랜치:** feat/75-maintenance-timeline



## 분석 질문과 범위

**소둔로의 기록된 정비 사유에 맞는 공정 변화가 관측되는가? 조치 전후 그 양상이 달라졌는가?**

`P1-CR1-CAL01` 정비 44건을 먼저 검토하고, 전후 관측이 있는 모든 작업을 상세 분석한다. 온도 차이가 가장 큰 센서를 감지 원인으로 고르지 않는다.
정비 사유·부품 → 검토할 신호 → 실제 관측 사실 → 관련성 해석 → 완료 후 변화 → 판단 한계 순서로 읽는다.
표와 그래프는 코드가 생성하며, 기존 `figures/jinny/06_maintenance_timeline_jinny`에 PNG로 자동 저장한다.

In [ ]:
import os
import pandas as pd

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
FIG_DIR = os.path.join(ROOT, "figures")
ENC = "utf-8-sig"

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")
BASE_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v1.csv")

from dataclasses import dataclass
from html import escape
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rcParams
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython import get_ipython
from IPython.display import display, HTML

if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')
for name in ['Malgun Gothic','AppleGothic','NanumGothic','Noto Sans CJK KR']:
    if name in {font.name for font in font_manager.fontManager.ttflist}:
        rcParams['font.family']=name
        break
rcParams.update({'axes.unicode_minus':False,'font.size':10,'figure.dpi':110})
FIGURE_DIR=os.path.join(FIG_DIR, 'jinny', '06_maintenance_timeline_jinny')
os.makedirs(FIGURE_DIR, exist_ok=True)
SAVED_FIGURES={}
EQP_TAG='P1-CR1-CAL01'
TC=[f'TC-Z{i}' for i in range(3,10)]+['TC-OUT1','TC-OUT2']
OP=[f'OP-Z{i}' for i in range(3,9)]
CP=['CP-Z4']
CHANNELS=TC+OP+CP
ZONE={**{f'TC-Z{i}':f'가열·균열 {i-2}존' for i in range(3,7)},
      'TC-Z7':'과시효대 1존','TC-Z8':'과시효대 2존','TC-Z9':'과시효대 슬러지 제거부',
      'TC-OUT1':'냉각대 1존','TC-OUT2':'냉각대 2존',
      **{f'OP-Z{i}':f'가열·균열 {i-2}존' for i in range(3,7)},
      'OP-Z7':'과시효대 1존','OP-Z8':'과시효대 2존','CP-Z4':'가열·균열존 분위기'}
UNIT={c:('℃' if c in TC else '%' if c in OP else ' (단위 미확인)') for c in CHANNELS}
POINTS={'DETC_DT':'감지 기록','STRT_DT':'정비 착수','END_DT':'정비 완료'}
EVENT_COLORS={'DETC_DT':'#b34e25','STRT_DT':'#74419b','END_DT':'#237658'}
TYPE_NAMES={'BM':'사후정비','CBM':'상태기반정비','TBM':'시간기준정비'}
TYPE_COLORS={'BM':'#b64d35','CBM':'#b87500','TBM':'#527a9b'}
COLORS=['#186d9b','#db7041','#288467','#9a579d','#b69027','#a55369']
GROUPS=[(TC[:4],'가열·균열존 | 1~4존','온도[℃]'),
        (TC[4:7],'과시효대 | 1·2존·슬러지 제거부','온도[℃]'),
        (TC[7:],'냉각대 | 1·2존','온도[℃]'),
        (OP,'가열·균열존·과시효대 | 제어출력','제어출력[%]'),
        (CP,'가열·균열존 | 분위기제어 CP','CP[단위 미확인]')]


@dataclass
class FurnaceData:
    equipment: pd.DataFrame
    maintenance: pd.DataFrame
    parts: pd.DataFrame
    raw: pd.DataFrame
    minutes: pd.DataFrame
    plans: pd.DataFrame
    summaries: pd.DataFrame
    lots: pd.DataFrame
    history: pd.DataFrame


In [ ]:
def load_furnace_data():
    master=pd.read_csv(EQP_CSV,encoding=ENC,dtype=str,keep_default_na=False)
    equipment=master.loc[master.EQP_TAG.eq(EQP_TAG)].copy()
    assert len(equipment)==1 and equipment.EQP_NM.str.contains('소둔').all()
    maintenance=pd.read_csv(MAINT_CSV,encoding=ENC,keep_default_na=False)
    maintenance=maintenance.loc[maintenance.EQP_TAG.eq(EQP_TAG)].copy()
    for field in POINTS:
        maintenance[field+'_RAW']=maintenance[field]
        maintenance[field]=pd.to_datetime(maintenance[field],errors='raise')
    maintenance=maintenance.sort_values(['DETC_DT','STRT_DT','WO_NO']).reset_index(drop=True)
    maintenance['WAIT_MIN']=(maintenance.STRT_DT-maintenance.DETC_DT).dt.total_seconds()/60
    maintenance['WORK_MIN']=(maintenance.END_DT-maintenance.STRT_DT).dt.total_seconds()/60
    changes=pd.read_csv(PARTCHG_CSV,encoding=ENC,dtype=str,keep_default_na=False)
    changes['SOURCE_ROW']=changes.index+2
    parts=changes.loc[changes.WO_NO.isin(maintenance.WO_NO)].merge(
        pd.read_csv(PART_CSV,encoding=ENC,dtype=str,keep_default_na=False)[['PART_CD','PART_NM']],
        on='PART_CD',how='left',validate='many_to_one')
    events=pd.read_csv(EVENT_CSV,encoding=ENC,dtype=str,keep_default_na=False)
    grade=events.loc[events.PLANT_CD.eq(equipment.PLANT_CD.iloc[0])&events.EVT_TYPE.eq('강종변경'),
                     ['EVT_DT','AFT_VAL']].rename(columns={'EVT_DT':'GRADE_DT','AFT_VAL':'GRADE'})
    grade['GRADE_DT']=pd.to_datetime(grade.GRADE_DT)
    raw=pd.read_csv(TEMP_CSV,encoding=ENC,usecols=['MEAS_DT','LOT_NO']+CHANNELS)
    raw['MEAS_DT']=pd.to_datetime(raw.MEAS_DT,errors='raise')
    raw['LOT_NO']=raw.LOT_NO.astype(str)
    raw=raw.sort_values('MEAS_DT').reset_index(drop=True)
    assert not raw.MEAS_DT.duplicated().any()
    raw=pd.merge_asof(raw,grade.sort_values('GRADE_DT'),left_on='MEAS_DT',right_on='GRADE_DT',direction='backward')
    raw['GRADE']=raw.GRADE.fillna('미확인')
    split=(raw.MEAS_DT.diff().dt.total_seconds().gt(5)|raw.LOT_NO.ne(raw.LOT_NO.shift())|
           raw.GRADE.ne(raw.GRADE.shift()))
    raw['SEGMENT']=split.cumsum();raw['MINUTE']=raw.MEAS_DT.dt.floor('min')
    keys=['LOT_NO','GRADE','SEGMENT','MINUTE'];groups=raw.groupby(keys,sort=False)
    minute=groups[CHANNELS].median().where(groups[CHANNELS].count().ge(30))
    minute['FIRST_DT']=groups.MEAS_DT.min();minute['LAST_DT']=groups.MEAS_DT.max()
    minute=minute.reset_index().sort_values('MINUTE').reset_index(drop=True)
    full=(minute.LAST_DT-minute.FIRST_DT).dt.total_seconds().ge(45)
    minute.loc[~full,CHANNELS]=np.nan
    minute['AVAILABLE_AT']=minute.MINUTE+pd.Timedelta(minutes=1)
    calendar=pd.read_csv(REPAIR_CSV,encoding=ENC,dtype=str,keep_default_na=False)
    plans=calendar.loc[calendar.PLANT_CD.eq(equipment.PLANT_CD.iloc[0])&calendar.SCOPE.str.contains('소둔',regex=False)].copy()
    plans['START']=pd.to_datetime(plans.STRT_DT)
    plans['END_EXCL']=pd.to_datetime(plans.END_DT)+pd.Timedelta(days=1)
    lots=raw.groupby('LOT_NO',sort=False).agg(START=('MEAS_DT','min'),END=('MEAS_DT','max'),
        ROWS=('MEAS_DT','size'),SEGMENTS=('SEGMENT','nunique'),GRADE=('GRADE',lambda x:' / '.join(x.unique())))
    data=FurnaceData(equipment,maintenance,parts,raw,minute,plans,pd.DataFrame(),lots,pd.DataFrame())
    data.summaries=summarize_observations(data)
    data.history=build_history(data)
    return data


In [ ]:
def summarize_observations(data):
    records=[]
    for segment,raw in data.raw.groupby('SEGMENT',sort=False):
        bins=data.minutes.loc[data.minutes.SEGMENT.eq(segment)]
        for channel in CHANNELS:
            valid=bins.loc[bins[channel].notna()]
            enough=len(valid)>=10
            first=valid[channel].head(5).median() if enough else np.nan
            last=valid[channel].tail(5).median() if enough else np.nan
            records.append(dict(SEGMENT=segment,LOT_NO=raw.LOT_NO.iloc[0],GRADE=raw.GRADE.iloc[0],
                START=raw.MEAS_DT.min(),END=raw.MEAS_DT.max(),CHANNEL=channel,ZONE=ZONE[channel],UNIT=UNIT[channel],
                BINS=len(valid),FIRST=first,LAST=last,DELTA=last-first,MIN=valid[channel].min(),MAX=valid[channel].max()))
    return pd.DataFrame(records)


def build_history(data):
    records=[]
    def add(start,end,kind,detail,key,wo='',lot='',grade='',precision='원본 시각'):
        records.append(dict(START=start,END=end,KIND=kind,DETAIL=detail,SOURCE_KEY=str(key),
                            WO_NO=wo,LOT_NO=lot,GRADE=grade,PRECISION=precision))
    for r in data.maintenance.itertuples():
        add(r.DETC_DT,r.DETC_DT,'감지 기록',f'{r.MNT_TYPE} · WO 비고 {r.REMARK!r} · 고장모드 {r.FAIL_MODE!r}',
            r.WO_NO+':DETC_DT',wo=r.WO_NO)
        add(r.STRT_DT,r.STRT_DT,'정비 착수',f'감지→착수 {r.WAIT_MIN:,.0f}분',r.WO_NO+':STRT_DT',wo=r.WO_NO)
        add(r.END_DT,r.END_DT,'정비 완료',f'작업 {r.WORK_MIN:,.0f}분 · 기록 정지길이 {r.DOWN_MIN:,.0f}분',
            r.WO_NO+':END_DT',wo=r.WO_NO)
        for part in data.parts.loc[data.parts.WO_NO.eq(r.WO_NO)].itertuples():
            add(r.STRT_DT,r.END_DT,'부품교체 기록',f'{part.PART_CD} {part.PART_NM} ×{part.QTY} / {part.CHG_RSN}',
                f'부품교체:{part.SOURCE_ROW}',wo=r.WO_NO,precision='교체시각 미기록; 연결 WO 작업 범위')
    for segment,g in data.raw.groupby('SEGMENT',sort=False):
        bins=data.minutes.loc[data.minutes.SEGMENT.eq(segment)]
        heat=bins[TC[:4]].median(axis=1).where(bins[TC[:4]].notna().all(axis=1)).dropna()
        text=f'LOT {g.LOT_NO.iloc[0]} · {len(g):,}개 측정 행'
        if len(heat)>=10:
            text+=f' · 가열 4존 중앙온도 초반/후반 각 5분창 {heat.head(5).median():.2f} → {heat.tail(5).median():.2f} ℃'
        add(g.MEAS_DT.min(),g.MEAS_DT.max(),'온도·출력 관측',text,f'관측구간:{segment}',
            lot=g.LOT_NO.iloc[0],grade=g.GRADE.iloc[0],precision='실제 측정 구간; 변화값은 초반·후반 각 5개 유효 분창 요약')
    for plan in data.plans.itertuples():
        add(plan.START,plan.END_EXCL,'소둔로 수리계획',plan.SCOPE,plan.PLAN_ID,
            precision='일자 계획·종료일 포함; 실제 정지시각 아님')
    return pd.DataFrame(records).sort_values(['START','END','KIND','SOURCE_KEY']).reset_index(drop=True)


def history_window(data,start=None,end=None):
    result=data.history.copy()
    if start is not None:
        start=pd.Timestamp(start)
        exclusive=result.KIND.isin(['부품교체 기록','소둔로 수리계획'])&result.START.ne(result.END)
        result=result.loc[result.END.ge(start)&~(exclusive&result.END.eq(start))]
    if end is not None:result=result.loc[result.START.lt(pd.Timestamp(end))]
    return result




PHASES=('감지 전','감지 후~정비 시작 전','정비 작업','완료 후')
PHASE_COLORS={'감지 전':'#eaf2f8','감지 후~정비 시작 전':'#fff0ce','정비 작업':'#eee3f4','완료 후':'#e3f1e9'}


def work_windows(work):
    return [(PHASES[0],work.DETC_DT-pd.Timedelta(hours=24),work.DETC_DT),
            (PHASES[1],work.DETC_DT,work.STRT_DT),(PHASES[2],work.STRT_DT,work.END_DT),
            (PHASES[3],work.END_DT,work.END_DT+pd.Timedelta(hours=24))]


def detection_identity(data,work):
    exact=data.raw.loc[data.raw.MEAS_DT.eq(work.DETC_DT)]
    if len(exact):
        row=exact.iloc[0]
        return dict(LOT_NO=row.LOT_NO,GRADE=row.GRADE,STATUS='감지시각 실측',MEAS_DT=row.MEAS_DT)
    before=data.raw.loc[data.raw.MEAS_DT.lt(work.DETC_DT)].tail(1)
    after=data.raw.loc[data.raw.MEAS_DT.gt(work.DETC_DT)].head(1)
    if len(before) and len(after):
        a,b=before.iloc[0],after.iloc[0]
        if a.SEGMENT==b.SEGMENT and (b.MEAS_DT-a.MEAS_DT).total_seconds()<=5:
            return dict(LOT_NO=a.LOT_NO,GRADE=a.GRADE,STATUS='감지 전후 5초 이내 동일 연속구간',MEAS_DT=a.MEAS_DT)
    return dict(LOT_NO='',GRADE='',STATUS='감지시각 측정 연결 불가',MEAS_DT=pd.NaT)


def phase_review(data,work):
    phases=[]
    for stage,lo,hi in work_windows(work):
        raw=data.raw.loc[data.raw.MEAS_DT.ge(lo)&data.raw.MEAS_DT.lt(hi)]
        bins=data.minutes.loc[data.minutes.MINUTE.ge(lo)&data.minutes.AVAILABLE_AT.le(hi)]
        phases.append(dict(STAGE=stage,START=lo,END=hi,ROWS=len(raw),
            BINS=int(bins[TC].notna().all(axis=1).sum()),FIRST=raw.MEAS_DT.min(),LAST=raw.MEAS_DT.max(),
            IDENTITIES=' / '.join(f'{lot} · {grade}' for lot,grade in raw[['LOT_NO','GRADE']].drop_duplicates().itertuples(index=False,name=None))))
    return pd.DataFrame(phases),pd.DataFrame()

In [ ]:
# These are review hypotheses, not a tag-to-failed-component mapping or an alarm rule.
EXTRA_FEATURES={'HEAT_LEVEL':('가열·균열 4존 중앙온도','℃'),
                'HEAT_SPREAD':('가열·균열 4존 최대-최소','℃'),
                'HEAT_OP':('가열·균열 4존 출력 중앙값','%')}
FEATURE_NAMES={c:f'{ZONE[c]} · {c}' for c in CHANNELS}
FEATURE_NAMES.update({k:v[0] for k,v in EXTRA_FEATURES.items()})
FEATURE_UNITS={**UNIT,**{k:v[1] for k,v in EXTRA_FEATURES.items()}}
WINDOWS=(5,15,30)
COMPARISONS={'DETC_DT':'감지 기록 앞뒤','STRT_DT':'착수 앞뒤','END_DT':'완료 앞뒤',
             'REPAIR_START':'착수 전 → 완료 후','REPAIR_DETC':'감지 기록 전 → 완료 후'}


def diagnostic_plan(data,work):
    parts=data.parts.loc[data.parts.WO_NO.eq(work.WO_NO)]
    names=' / '.join(parts.PART_NM.fillna(''))
    remark=' '.join(work.REMARK.split())
    action=' / '.join(f'{p.PART_CD} {p.PART_NM} ×{p.QTY} ({p.CHG_RSN})' for p in parts.itertuples())
    flags=[]
    thermal=['HEAT_LEVEL','HEAT_SPREAD','HEAT_OP','TC-Z7','TC-Z8','TC-Z9','TC-OUT1','TC-OUT2','CP-Z4']
    if work.MNT_TYPE=='TBM':
        category='계획·시간기준점검';primary='HEAT_LEVEL';features=thermal
        question='계획점검 전후 운전 양상이 어떻게 달라졌는가?'
        missing='고장 발생으로 해석하지 않음; 목표 온도·운전 상태 필요'
        if work.FAIL_MODE or any(s in remark for s in ['마모','알람','트립','불량','과열','절연']):
            flags.append('TBM 유형과 고장성 비고·모드의 의미 확인 필요')
    elif '가스 분석계' in names:
        category='분위기 계측기 교체';primary='CP-Z4';features=['CP-Z4','HEAT_LEVEL','HEAT_SPREAD','HEAT_OP']
        question='분석계 셀 교체와 관련해 CP 수준·변동 폭의 차이가 관측되는가?'
        missing='CP는 간접지표; 분석계 지시값·교정 결과·설치 존 필요'
        if work.FAIL_MODE=='WER':flags.append('마모 기록과 분석계 셀 교체의 대상 연결 미기록')
    elif work.FAIL_MODE=='ELE' or any(s in remark for s in ['절연','인버터','과열','전기']):
        category='전기·구동 현상';primary='HEAT_LEVEL'
        features=[x for i in range(3,9) for x in [f'TC-Z{i}',f'OP-Z{i}']]+['HEAT_LEVEL','HEAT_SPREAD']
        question='같은 존의 온도·출력 변화가 함께 관측되는가?'
        missing='출력은 전류·절연값이 아님; 전기 계측·구동 알람·부품 위치 필요'
        if any(s in names for s in ['오일씰','O링']):flags.append('전기 현상과 실링 교체의 대상 연결 미기록')
    elif work.FAIL_MODE=='MAL' or any(s in remark for s in ['트립','동작','오작동']):
        category='동작·트립 현상';primary='HEAT_LEVEL'
        features=[x for i in range(3,9) for x in [f'TC-Z{i}',f'OP-Z{i}']]+['HEAT_LEVEL','HEAT_SPREAD']
        question='출력 변화와 온도 응답이 동반되는가?'
        missing='운전/정지 상태·모터 전류·회전수·트립 코드 필요'
    elif work.FAIL_MODE=='WER' or '마모' in remark:
        category='마모 현상';primary='HEAT_LEVEL';features=thermal
        question='마모 기록 전후 공정 운전 변화가 관측되는가?'
        missing='온도·출력은 간접지표; 마모량·대상 부품·위치 필요'
    else:
        category='현상·대상 미특정';primary='HEAT_LEVEL';features=thermal
        question='기록된 현상과 연결할 수 있는 공정 변화가 있는가?'
        missing='고장 현상·대상 부품·위치를 먼저 확인해야 함'
    return dict(category=category,question=question,primary=primary,features=features,missing=missing,
                flags=flags,action=action or '부품교체 기록 없음; 수행 조치의 상세 내용 미기록')


def signal_bins(data):
    result=data.minutes.copy()
    heat=result[TC[:4]];out=result[OP[:4]]
    valid=heat.notna().all(axis=1)
    result['HEAT_LEVEL']=heat.median(axis=1).where(valid)
    result['HEAT_SPREAD']=(heat.max(axis=1)-heat.min(axis=1)).where(valid)
    result['HEAT_OP']=out.median(axis=1).where(out.notna().all(axis=1))
    return result


def edge_run(group,key,side):
    valid=group.loc[group[key].notna()].sort_values('MINUTE')
    if valid.empty:return valid
    runs=(valid.MINUTE.diff().ne(pd.Timedelta(minutes=1))|valid.SEGMENT.ne(valid.SEGMENT.shift())).cumsum()
    return valid.loc[runs.eq(runs.iloc[-1] if side=='before' else runs.iloc[0])]


def signal_stats(data,group,key):
    values=group[key]
    result={'MEDIAN':values.median(),'IQR':values.quantile(.75)-values.quantile(.25),
            'STEP':values.diff().abs().median(),'ZERO':np.nan,'HUNDRED':np.nan}
    if key in OP:
        raw=data.raw.loc[data.raw.SEGMENT.eq(group.SEGMENT.iloc[0])&
            data.raw.MEAS_DT.ge(group.FIRST_DT.min())&data.raw.MEAS_DT.le(group.LAST_DT.max()),key].dropna()
        if len(raw):result.update(ZERO=raw.eq(0).mean()*100,HUNDRED=raw.eq(100).mean()*100)
    return result


def compare_windows(data,work,bins):
    records=[]
    for code in COMPARISONS:
        leftpoint=work.STRT_DT if code=='REPAIR_START' else work.DETC_DT if code=='REPAIR_DETC' else getattr(work,code)
        rightpoint=work.END_DT if code.startswith('REPAIR_') else leftpoint
        for width in WINDOWS:
            left=bins.loc[bins.MINUTE.ge(leftpoint-pd.Timedelta(minutes=width))&bins.AVAILABLE_AT.le(leftpoint)]
            right=bins.loc[bins.MINUTE.ge(rightpoint)&bins.AVAILABLE_AT.le(rightpoint+pd.Timedelta(minutes=width))]
            for (lot,grade),before in left.groupby(['LOT_NO','GRADE'],sort=False):
                after=right.loc[right.LOT_NO.eq(lot)&right.GRADE.eq(grade)]
                if after.empty:continue
                before=before.loc[before.SEGMENT.eq(before.iloc[-1].SEGMENT)]
                after=after.loc[after.SEGMENT.eq(after.iloc[0].SEGMENT)]
                if not code.startswith('REPAIR_') and before.SEGMENT.iloc[0]!=after.SEGMENT.iloc[0]:continue
                for key in CHANNELS+list(EXTRA_FEATURES):
                    a,b=edge_run(before,key,'before'),edge_run(after,key,'after')
                    if min(len(a),len(b))<3:continue
                    x,y=signal_stats(data,a,key),signal_stats(data,b,key)
                    records.append(dict(CODE=code,POINT=COMPARISONS[code],WINDOW=width,KEY=key,LOT_NO=lot,GRADE=grade,
                        BEFORE=x['MEDIAN'],AFTER=y['MEDIAN'],DELTA=y['MEDIAN']-x['MEDIAN'],
                        IQR_BEFORE=x['IQR'],IQR_AFTER=y['IQR'],STEP_BEFORE=x['STEP'],STEP_AFTER=y['STEP'],
                        ZERO_BEFORE=x['ZERO'],ZERO_AFTER=y['ZERO'],HUNDRED_BEFORE=x['HUNDRED'],HUNDRED_AFTER=y['HUNDRED'],
                        BEFORE_BINS=len(a),AFTER_BINS=len(b),COVERAGE_OK=min(len(a),len(b))>=int(np.ceil(width*.8)),
                        BEFORE_START=a.FIRST_DT.min(),BEFORE_END=a.LAST_DT.max(),AFTER_START=b.FIRST_DT.min(),AFTER_END=b.LAST_DT.max(),
                        BEFORE_SEGMENT=int(a.SEGMENT.iloc[0]),AFTER_SEGMENT=int(b.SEGMENT.iloc[0]),
                        CONTINUITY='같은 연속구간' if a.SEGMENT.iloc[0]==b.SEGMENT.iloc[0] else '비연속·보간 없음'))
    return pd.DataFrame(records)


def phase_signals(data,work,bins):
    rows=[]
    for stage,lo,hi in work_windows(work):
        selected=bins.loc[bins.MINUTE.ge(lo)&bins.AVAILABLE_AT.le(hi)]
        for segment,group in selected.groupby('SEGMENT',sort=False):
            for key in CHANNELS+list(EXTRA_FEATURES):
                valid=group.loc[group[key].notna()]
                runs=valid.MINUTE.diff().ne(pd.Timedelta(minutes=1)).cumsum()
                for run,part in valid.groupby(runs,sort=False):
                    if len(part)<3:continue
                    stat=signal_stats(data,part,key)
                    first=part[key].head(5).median() if len(part)>=10 else np.nan
                    last=part[key].tail(5).median() if len(part)>=10 else np.nan
                    rows.append(dict(STAGE=stage,KEY=key,LOT_NO=part.LOT_NO.iloc[0],GRADE=part.GRADE.iloc[0],
                        SEGMENT=segment,RUN=run,START=part.FIRST_DT.min(),END=part.LAST_DT.max(),BINS=len(part),
                        **stat,MIN=part[key].min(),MAX=part[key].max(),FIRST=first,LAST=last,DELTA=last-first))
    return pd.DataFrame(rows)


def matching_rows(review,code,key=None,width=15):
    table=review['evidence']
    if table.empty:return table
    return table.loc[table.CODE.eq(code)&table.WINDOW.eq(width)&table.KEY.eq(key or review['plan']['primary'])]


def value_text(key,value,difference=False):
    if pd.isna(value):return '—'
    unit=FEATURE_UNITS[key]
    if key in CP and value!=0 and abs(value)<.0001:return f'{value:+.2e}' if difference else f'{value:.2e}'
    return f'{value:+.4f}' if difference and key in CP else f'{value:.4f}' if key in CP else f'{value:+.2f}' if difference else f'{value:.2f}'



def output_zero_episodes(data,work,raw):
    """Observed exact-zero runs; break at every missing second, stage boundary and LOT/grade segment."""
    if raw.empty:return pd.DataFrame()
    raw=raw.copy();raw['STAGE']=''
    for stage,lo,hi in work_windows(work):raw.loc[raw.MEAS_DT.ge(lo)&raw.MEAS_DT.lt(hi),'STAGE']=stage
    records=[]
    for channel in OP:
        zero=raw[channel].eq(0)
        breaks=(~zero|~zero.shift(fill_value=False)|raw.MEAS_DT.diff().ne(pd.Timedelta(seconds=1))|
                raw.SEGMENT.ne(raw.SEGMENT.shift())|raw.STAGE.ne(raw.STAGE.shift())).cumsum()
        for _,group in raw.loc[zero].groupby(breaks[zero],sort=False):
            if len(group)<2:continue
            tc=channel.replace('OP-','TC-')
            records.append(dict(STAGE=group.STAGE.iloc[0],CHANNEL=channel,TC=tc,LOT_NO=group.LOT_NO.iloc[0],GRADE=group.GRADE.iloc[0],
                SEGMENT=int(group.SEGMENT.iloc[0]),START=group.MEAS_DT.min(),END=group.MEAS_DT.max(),SAMPLES=len(group),
                SPAN_SEC=(group.MEAS_DT.max()-group.MEAS_DT.min()).total_seconds(),
                TEMP_MIN=group[tc].min(),TEMP_MAX=group[tc].max(),CP_MIN=group['CP-Z4'].min(),CP_MAX=group['CP-Z4'].max()))
    return pd.DataFrame(records)


def work_review(data,work,bins=None):
    bins=signal_bins(data) if bins is None else bins
    phases,metrics=phase_review(data,work)
    lo,hi=work_windows(work)[0][1],work_windows(work)[-1][2]
    raw=data.raw.loc[data.raw.MEAS_DT.ge(lo)&data.raw.MEAS_DT.lt(hi)]
    overlaps=[]
    for other in data.maintenance.loc[data.maintenance.WO_NO.ne(work.WO_NO)].itertuples():
        a,b=max(work.STRT_DT,other.STRT_DT),min(work.END_DT,other.END_DT)
        if a<b:overlaps.append(f'{other.WO_NO} 작업 겹침 {a:%m-%d %H:%M}~{b:%H:%M}')
        if ((raw.MEAS_DT.ge(work.STRT_DT-pd.Timedelta(minutes=30)))&(raw.MEAS_DT.lt(work.STRT_DT))&
            (raw.MEAS_DT.ge(other.STRT_DT))&(raw.MEAS_DT.lt(other.END_DT))).any():
            overlaps.append(f'착수 전 비교 관측에도 {other.WO_NO} 작업 진행')
        if ((raw.MEAS_DT.ge(work.END_DT))&(raw.MEAS_DT.ge(other.STRT_DT))&(raw.MEAS_DT.lt(other.END_DT))).any():
            overlaps.append(f'완료 후 관측에도 {other.WO_NO} 작업 진행')
    review=dict(work=work,identity=detection_identity(data,work),phases=phases,metrics=metrics,raw=raw,
                plan=diagnostic_plan(data,work),evidence=compare_windows(data,work,bins),
                signals=phase_signals(data,work,bins),overlaps=overlaps,episodes=output_zero_episodes(data,work,raw))
    review['candidates']=observation_candidates(data,work,raw,review['plan'])
    review['detection_verdict']=('계획점검: 고장 감지로 해석하지 않음' if work.MNT_TYPE=='TBM' else
        '감지 앞뒤 운전 관측 가능; 직접 고장 신호 미보유' if review['identity']['LOT_NO'] else '감지시각의 직접 관측 연결 불가')
    repair=matching_rows(review,'REPAIR_START')
    review['effect_verdict']='같은 LOT 운전 차이 비교 가능; 개선 판정 보류' if len(repair) else '착수 전·완료 후 동일 LOT 비교 불가'
    return review


def build_reviews(data):
    bins=signal_bins(data);reviews={};rows=[]
    for work in data.maintenance.itertuples():
        r=work_review(data,work,bins);reviews[work.WO_NO]=r
        row={'WO':work.WO_NO,'유형':work.MNT_TYPE,'감지기록':work.DETC_DT,'검토 분류':r['plan']['category'],
             '기록된 현상':work.REMARK,'기록된 조치':r['plan']['action'],
             '감지 LOT·강종':f'{r["identity"]["LOT_NO"]} / {r["identity"]["GRADE"]}' if r['identity']['LOT_NO'] else '연결 불가',
             '감지 판단':r['detection_verdict'],'완료 후 판단':r['effect_verdict'],
             '추가 확인':r['plan']['missing'],'기록 검토':' / '.join(r['plan']['flags']),
             'OBSERVED':bool(len(r['raw']))}
        for phase in r['phases'].itertuples():row[phase.STAGE]=phase.BINS
        rows.append(row)
    return reviews,pd.DataFrame(rows)


def work_findings(review):
    plan=review['plan'];key=plan['primary'];unit=FEATURE_UNITS[key];label=FEATURE_NAMES[key];lines=[]
    rows=matching_rows(review,'DETC_DT',key)
    if review['work'].MNT_TYPE=='TBM':
        prefix='계획점검 기록시각 앞뒤'
    else:prefix='감지 기록 앞뒤'
    if len(rows):
        r=rows.iloc[0]
        lines.append(f'{prefix}: {label} {value_text(key,r.BEFORE)} → {value_text(key,r.AFTER)} {unit}; '
            f'전/후 {r.BEFORE_BINS}/{r.AFTER_BINS}분창. 실제 감지 알람과의 대응은 미확인.')
    else:lines.append('감지시각의 같은 LOT 전후 비교 불가. 이전 관측을 감지 순간의 현상으로 연결하지 않음.')
    profiles=review['signals']
    during=profiles.loc[profiles.STAGE.eq('정비 작업')&profiles.KEY.eq(key)] if len(profiles) else pd.DataFrame()
    if len(during):
        r=during.loc[during.BINS.idxmax()]
        trend=f'초반/후반 {value_text(key,r.FIRST)} → {value_text(key,r.LAST)}' if pd.notna(r.DELTA) else f'중앙값 {value_text(key,r.MEDIAN)}'
        lines.append(f'작업 중 LOT {r.LOT_NO}: {label} {trend} {unit}; 분창간 IQR {value_text(key,r.IQR)}. 실제 정지구간은 미상.')
    else:lines.append('작업 중 관측은 없어 해당 단계의 공정 양상·정지 양상을 확인할 수 없음.')
    repairs=matching_rows(review,'REPAIR_START',key)
    if len(repairs):
        r=repairs.iloc[0]
        lines.append(f'착수 전/완료 후 각 15분: {label} {value_text(key,r.BEFORE)} → {value_text(key,r.AFTER)} {unit}; '
                     f'IQR {value_text(key,r.IQR_BEFORE)} → {value_text(key,r.IQR_AFTER)}. 정상화·조치 효과 판정은 보류.')
    else:lines.append('착수 전과 완료 후의 같은 LOT 비교가 불가해 조치 전후 변화는 검증하지 못함.')
    sensitivity=review['evidence']
    if len(sensitivity):
        code='REPAIR_START' if len(repairs) else 'DETC_DT'
        sensitivity=sensitivity.loc[sensitivity.CODE.eq(code)&sensitivity.KEY.eq(key)&sensitivity.COVERAGE_OK]
        if len(sensitivity):
            txt=' / '.join(f'{r.WINDOW}분 {value_text(key,r.DELTA,True)}' for r in sensitivity.itertuples())
            direction='차이 방향이 창에 따라 바뀜' if sensitivity.DELTA.min()<0<sensitivity.DELTA.max() else '표시한 창에서 차이 방향 동일' if len(sensitivity)>1 else '충분 관측 창 1개'
            lines.append(f'창 민감도: {txt} {unit}; {direction}. 알람·정상 기준이 아님.')
    while len(lines)<4:lines.append(plan['missing'] if len(lines)==3 else '')
    return lines[:4]


def case_interpretation(review):
    category=review['plan']['category'];profile=review['signals'];identity=review['identity']
    if category=='계획·시간기준점검':
        during=profile.loc[profile.STAGE.eq('정비 작업')&profile.KEY.isin(TC[:4])].dropna(subset=['DELTA']) if len(profile) else pd.DataFrame()
        if len(during):
            for _,g in during.groupby(['LOT_NO','GRADE','SEGMENT','RUN'],sort=False):
                if set(g.KEY)==set(TC[:4]) and g.START.nunique()==g.END.nunique()==1 and g.DELTA.gt(0).all():
                    return f'가열 4존 모두 작업 초반보다 후반 높음 (각 존 Δ {g.DELTA.min():+.2f}~{g.DELTA.max():+.2f}℃). 승온·목표 변경 여부를 확인해야 하며 정상화로 판정하지 않음.'
        return '시간기준점검 전후의 운전 관측이다. 감지 기록을 고장 발생이나 이상 징후의 정답으로 사용하지 않음.'
    if category=='분위기 계측기 교체':
        return 'CP는 분석계 교체의 간접 검토 신호다. 마모 부위·분석계 지시값·교정 결과가 없어 마모 또는 분석계 이상으로 판정하지 못함.'
    if category=='마모 현상':return '마모량·대상 부품이 없어 공정 변화와 마모의 관련성을 검증하지 못함.'
    if category in ['전기·구동 현상','동작·트립 현상']:
        return ('감지 순간 측정이 없으므로 앞선 출력 변동·온도 응답을 해당 고장 징후로 연결할 수 없음.' if not identity['LOT_NO'] else
                '온도·출력은 운전 반응이다. 전기·동작 이상 판정에는 직접 계측과 운전/트립 상태가 필요함.')
    return '정비 현상·대상이 특정되지 않아 센서 변화와 해당 정비의 관련성을 검증하지 못함.'


def episode_finding(review):
    episodes=review['episodes']
    if episodes.empty:return ''
    r=episodes.loc[episodes.SPAN_SEC.idxmax()]
    return (f'실측 출력0 최장 구간: LOT {r.LOT_NO} / {r.GRADE} · {r.CHANNEL} ({ZONE[r.CHANNEL]}) '
        f'{r.START:%m-%d %H:%M:%S}~{r.END:%H:%M:%S}, 연속 {r.SAMPLES}개 1초 표본; '
        f'{r.TC} {r.TEMP_MIN:.2f}~{r.TEMP_MAX:.2f}℃. 설비 정지·고장 판정은 아님.')


def elapsed_text(delta):
    seconds=max(0,int(delta.total_seconds()));minutes=seconds//60
    if minutes==0:return f'{seconds}초'
    days,remaining=divmod(minutes,1440);hours,mins=divmod(remaining,60)
    return (f'{days}일 ' if days else '')+(f'{hours}시간 ' if hours else '')+(f'{mins}분' if mins or not (days or hours) else '')


def interval_stage(work,start,end):
    names=[stage for stage,lo,hi in work_windows(work) if end>=lo and start<hi]
    return names[0] if len(names)==1 else '단계 경계 포함'


def observation_candidates(data,work,raw,plan):
    """Select display windows, not fault alarms: consecutive ten-minute paired changes, no normal threshold."""
    if raw.empty:return pd.DataFrame()
    records=[]
    for segment,part in raw.groupby('SEGMENT',sort=False):
        bins=data.minutes.loc[data.minutes.SEGMENT.eq(segment)&data.minutes.FIRST_DT.ge(part.MEAS_DT.min())&
                              data.minutes.LAST_DT.le(part.MEAS_DT.max())]
        # Prefer pre-detection windows; planned inspections use pre-work windows.
        point=work.STRT_DT if work.MNT_TYPE=='TBM' else work.DETC_DT
        earlier=bins.loc[bins.AVAILABLE_AT.le(point)]
        if len(earlier)>=10:bins=earlier
        keys=[('CP-Z4',None)] if plan['primary']=='CP-Z4' else [(f'TC-Z{i}',f'OP-Z{i}') for i in range(3,9)]
        for tc,op in keys:
            valid=bins.dropna(subset=[tc]+([op] if op else []))
            runs=valid.MINUTE.diff().ne(pd.Timedelta(minutes=1)).cumsum()
            choices=[]
            for _,group in valid.groupby(runs,sort=False):
                if len(group)<10:continue
                before=group[tc].rolling(5).median().shift(5);after=group[tc].rolling(5).median()
                if op:
                    out_before=group[op].rolling(5).median().shift(5);out_after=group[op].rolling(5).median()
                    score=(after-before).abs()*(out_after-out_before).abs()
                else:
                    out_before=out_after=pd.Series(np.nan,index=group.index);score=(after-before).abs()
                if not score.notna().any():continue
                idx=score.idxmax();pos=group.index.get_loc(idx);window=group.iloc[pos-9:pos+1]
                choices.append(dict(SEGMENT=segment,LOT_NO=window.LOT_NO.iloc[0],GRADE=window.GRADE.iloc[0],
                    START=window.FIRST_DT.min(),END=window.LAST_DT.max(),MIDPOINT=window.MINUTE.iloc[5],
                    TC=tc,OP=op,BEFORE=before.loc[idx],AFTER=after.loc[idx],DELTA=after.loc[idx]-before.loc[idx],
                    OP_BEFORE=out_before.loc[idx],OP_AFTER=out_after.loc[idx],OP_DELTA=out_after.loc[idx]-out_before.loc[idx],
                    SCORE=float(score.loc[idx]),STAGE=interval_stage(work,window.FIRST_DT.min(),window.LAST_DT.max())))
            if choices:records.append(max(choices,key=lambda row:row['SCORE']))
    return pd.DataFrame(records)


def block_candidate(review,block=None):
    rows=review['candidates']
    if rows.empty:return None
    if block is not None:rows=rows.loc[rows.START.ge(block['start'])&rows.END.le(block['end'])]
    return None if rows.empty else rows.loc[rows.SCORE.idxmax()]


def short_zone(channel):
    if channel=='CP-Z4':return '가열·균열존 분위기'
    number=int(channel.split('Z')[-1])
    return f'가열·균열 {number-2}존' if number<=6 else f'과시효대 {number-6}존'


def candidate_brief(row):
    if row is None:return '10분 연속 비교자료 부족'
    text=f'{short_zone(row.TC)} · {row.TC} {value_text(row.TC,row.DELTA,True)}{FEATURE_UNITS[row.TC]}'
    if row.OP:text+=f' / {row.OP} {row.OP_DELTA:+.2f}%p'
    return text


def repair_available(review):
    return bool(len(matching_rows(review,'REPAIR_START')))


def relation_summary(review):
    work=review['work'];raw=review['raw']
    if raw.empty:return '관측 연결 없음','전후 비교 불가'
    if repair_available(review):return '동일 LOT·강종 전후 비교 가능','운전 차이 확인 · 조치 효과 별도 검토'
    post=raw.loc[raw.MEAS_DT.ge(work.END_DT)]
    if len(post):return '완료 후 관측만 있음',f'완료 {elapsed_text(post.MEAS_DT.min()-work.END_DT)} 후부터 측정'
    if review['identity']['LOT_NO']:return '감지 앞뒤 비교 가능','정비 완료 후 자료 없음'
    before=raw.loc[raw.MEAS_DT.lt(work.DETC_DT)]
    if len(before):return f'감지 {elapsed_text(work.DETC_DT-before.MEAS_DT.max())} 전 관측 종료','정비 완료 후 자료 없음'
    return '감지 이후의 관측만 있음','정비 완료 후 자료 없음'


def concise_conclusion(review):
    raw=review['raw'];work=review['work'];plan=review['plan'];row=block_candidate(review)
    if repair_available(review):
        if work.MNT_TYPE=='TBM' and review['overlaps']:
            return '가열 4존 공통 승온·다른 정비 중첩 → 같은 LOT의 전후 차이는 보이지만 개별 정비 효과는 분리 불가.'
        sensitivity=review['evidence'];s=sensitivity.loc[sensitivity.CODE.eq('REPAIR_START')&sensitivity.KEY.eq(plan['primary'])&sensitivity.COVERAGE_OK]
        if len(s) and s.DELTA.min()<0<s.DELTA.max():
            return '가열 4존 중앙온도의 전후 차이가 5·15·30분 창에 따라 방향이 바뀜 → 마모 개선으로 판정할 근거 부족.'
        return '같은 LOT·강종의 전후 운전 차이는 비교 가능하며, 목표값·운전 상태 확인 전에는 정비 효과를 판정하지 않음.'
    if raw[['LOT_NO','GRADE']].drop_duplicates().shape[0]>1:
        return 'LOT 37/SPCE → 38/SPCEN은 서로 다른 제품 조건의 정비 시작 전 관측 → 정비 이후 변화는 비교 불가.' if set(raw.LOT_NO)=={'240037','240038'} else '서로 다른 LOT·강종의 관측 → 정비 전후 효과 비교에 사용하지 않음.'
    if review['identity']['LOT_NO'] and plan['primary']=='CP-Z4':
        rows=matching_rows(review,'DETC_DT','CP-Z4')
        if len(rows):
            r=rows.iloc[0]
            return f'감지 앞뒤 CP {r.BEFORE:.4f}→{r.AFTER:.4f} 관측 → 마모 원인은 미확인, 완료 후 측정이 없어 교체 효과 비교 불가.'
    first,last=relation_summary(review)
    if raw.MEAS_DT.ge(work.END_DT).any():return first+' · '+last+' → 정비 이전 자료가 없어 개선 여부는 비교 불가.'
    noun='계획점검 기록' if work.MNT_TYPE=='TBM' else '실제 감지'
    return f'{candidate_brief(row)} 관측 → {first}; {noun} 원인·정비 이후 변화는 확인 불가.'

In [ ]:
def show_table(frame,title,height=360,collapsed=False):
    if frame.empty:return
    body=f'<div style="max-height:{int(height)}px;overflow:auto">'+frame.to_html(index=False,escape=True,border=0,classes='furnace-table')+'</div>'
    heading=f'{escape(title)} · {len(frame)}행'
    display(HTML(f'<details><summary>{heading}</summary>{body}</details>' if collapsed else f'<h4>{heading}</h4>{body}'))


def show_history(frame,title='CAL01 시간순 사건·조치·관측',collapsed=True):
    table=frame.copy()
    table['시간·구간']=[f'{r.START:%m-%d %H:%M:%S}'+(f' ~ {r.END:%m-%d %H:%M:%S}' if r.END>r.START else '') for r in table.itertuples()]
    table=table.rename(columns={'KIND':'기록','DETAIL':'사건·조치·관측 변화','WO_NO':'정비 WO','LOT_NO':'LOT','GRADE':'강종(CR1 연결)','PRECISION':'시각 해석'})
    show_table(table[['시간·구간','기록','사건·조치·관측 변화','정비 WO','LOT','강종(CR1 연결)','시각 해석']],title,collapsed=collapsed)


def show_audit(audit):
    cols=['WO','유형','검토 분류','기록된 현상','감지 LOT·강종']+list(PHASES)+['완료 후 판단']
    show_table(audit.loc[audit.OBSERVED,cols], '관측이 있는 모든 작업 · 단계별 수치는 유효 온도 1분창 수',height=380)
    show_table(audit.drop(columns='OBSERVED'),'전체 44건의 질문·조치·자료 범위·판단',height=380,collapsed=True)
    flagged=audit.loc[audit['기록 검토'].ne(''),['WO','유형','기록된 현상','기록된 조치','기록 검토']]
    show_table(flagged,'원문에서 대상 연결·정비유형을 확인해야 할 기록',collapsed=True)


def show_evidence(review):
    candidates=review['candidates']
    if len(candidates):
        rows=[]
        for r in candidates.itertuples():
            rows.append({'LOT / 연결 강종':f'{r.LOT_NO} / {r.GRADE}','단계':r.STAGE,
                '관측변화 창':f'{r.START:%m-%d %H:%M}~{r.END:%H:%M:%S}','존':short_zone(r.TC),'온도/CP':r.TC,
                '앞5분→뒤5분':f'{value_text(r.TC,r.BEFORE)} → {value_text(r.TC,r.AFTER)}',
                '출력':r.OP or '', '출력 앞5분→뒤5분':f'{r.OP_BEFORE:.2f} → {r.OP_AFTER:.2f} %' if r.OP else ''})
        show_table(pd.DataFrame(rows),'모든 존의 관측변화 후보 · 그래프 확대창 선정용이며 실제 감지 알람은 아님',collapsed=True)
    evidence=review['evidence'];features=review['plan']['features']
    if len(evidence):
        rows=[]
        for r in evidence.loc[evidence.WINDOW.eq(15)&evidence.KEY.isin(features)].itertuples():
            key=r.KEY;unit=FEATURE_UNITS[key];difference_unit='%p' if unit=='%' else unit
            times=f'{r.BEFORE_START:%m-%d %H:%M:%S}~{r.BEFORE_END:%H:%M:%S} / {r.AFTER_START:%m-%d %H:%M:%S}~{r.AFTER_END:%H:%M:%S}'
            rows.append({'비교':r.POINT,'LOT / 연결 강종':f'{r.LOT_NO} / {r.GRADE}','신호':FEATURE_NAMES[key],
                '전 / 후 실제 관측':times,'분창 전/후':f'{r.BEFORE_BINS}/{r.AFTER_BINS}',
                '수준 전→후':f'{value_text(key,r.BEFORE)} → {value_text(key,r.AFTER)} {unit}',
                '수준 차이':f'{value_text(key,r.DELTA,True)} {difference_unit}',
                'IQR 전→후':f'{value_text(key,r.IQR_BEFORE)} → {value_text(key,r.IQR_AFTER)} {difference_unit}',
                '연속 1분 변화 전→후':f'{value_text(key,r.STEP_BEFORE)} → {value_text(key,r.STEP_AFTER)} {difference_unit}',
                '원본 출력0 전→후':f'{r.ZERO_BEFORE:.2f} → {r.ZERO_AFTER:.2f} %' if pd.notna(r.ZERO_BEFORE) else '',
                '원본 출력100 전→후':f'{r.HUNDRED_BEFORE:.2f} → {r.HUNDRED_AFTER:.2f} %' if pd.notna(r.HUNDRED_BEFORE) else '',
                '관측 조건':('양쪽 80% 이상' if r.COVERAGE_OK else '일부 관측')+' · '+r.CONTINUITY})
        show_table(pd.DataFrame(rows),'15분 비교 · 수준·변동 폭·국소 변화와 원본 출력 비중',collapsed=True)
        sensitivity=evidence.loc[evidence.KEY.eq(review['plan']['primary'])].copy()
        if len(sensitivity):
            unit=FEATURE_UNITS[review['plan']['primary']]
            for col in ['BEFORE','AFTER','DELTA','IQR_BEFORE','IQR_AFTER','STEP_BEFORE','STEP_AFTER']:
                sensitivity[col]=sensitivity[col].map(lambda v:value_text(review['plan']['primary'],v,col=='DELTA'))
            sensitivity=sensitivity[['POINT','WINDOW','LOT_NO','GRADE','BEFORE','AFTER','DELTA','IQR_BEFORE','IQR_AFTER','BEFORE_BINS','AFTER_BINS','COVERAGE_OK']]
            sensitivity=sensitivity.rename(columns={'POINT':'비교','WINDOW':'창 길이(분)','GRADE':'강종(CR1 연결)','BEFORE':'이전 수준','AFTER':'이후 수준','DELTA':'차이',
                'IQR_BEFORE':'이전 IQR','IQR_AFTER':'이후 IQR','BEFORE_BINS':'이전 분창','AFTER_BINS':'이후 분창','COVERAGE_OK':'양쪽 80% 관측'})
            show_table(sensitivity,f'5·15·30분 창 민감도 · {FEATURE_NAMES[review["plan"]["primary"]]} [{unit}]',collapsed=True)
    episodes=review['episodes']
    if len(episodes):
        longest=episodes.sort_values('SPAN_SEC').groupby(['STAGE','CHANNEL','LOT_NO','GRADE'],sort=False).tail(1).copy()
        longest['센서 존']=longest.CHANNEL.map(ZONE)
        longest['동시 온도 범위']=longest.apply(lambda r:f'{r.TC}: {r.TEMP_MIN:.2f}~{r.TEMP_MAX:.2f}℃',axis=1)
        longest['동시 CP 범위']=longest.apply(lambda r:f'{r.CP_MIN:.4f}~{r.CP_MAX:.4f}',axis=1)
        longest=longest.rename(columns={'STAGE':'단계','CHANNEL':'출력 센서','GRADE':'강종(CR1 연결)','START':'관측 시작','END':'관측 끝','SAMPLES':'연속1초 표본 수','SPAN_SEC':'첫끝 측정 간격(초)'})
        show_table(longest[['단계','LOT_NO','강종(CR1 연결)','센서 존','출력 센서','관측 시작','관측 끝','연속1초 표본 수','첫끝 측정 간격(초)','동시 온도 범위','동시 CP 범위']],
                   '단계·존별 최장 실측 출력0 구간 · 1초 공백도 연결하지 않음 · 설비 정지 판정 아님',collapsed=True)
    profiles=review['signals']
    if len(profiles):
        rows=[]
        for r in profiles.loc[profiles.KEY.isin(features)].itertuples():
            key=r.KEY;unit=FEATURE_UNITS[key]
            rows.append({'단계':r.STAGE,'LOT / 연결 강종':f'{r.LOT_NO} / {r.GRADE}','신호':FEATURE_NAMES[key],
                '관측':f'{r.START:%m-%d %H:%M:%S}~{r.END:%H:%M:%S}','분창':r.BINS,
                '중앙값':f'{value_text(key,r.MEDIAN)} {unit}','범위':f'{value_text(key,r.MIN)} ~ {value_text(key,r.MAX)} {unit}',
                'IQR':value_text(key,r.IQR),'연속1분 변화':value_text(key,r.STEP),
                '초반5→후반5':f'{value_text(key,r.FIRST)} → {value_text(key,r.LAST)}',
                '원본 출력0':f'{r.ZERO:.2f} %' if pd.notna(r.ZERO) else '',
                '원본 출력100':f'{r.HUNDRED:.2f} %' if pd.notna(r.HUNDRED) else ''})
        show_table(pd.DataFrame(rows),'단계별 연속구간의 양상 · 초반/후반은 최소 10분창일 때만 계산',collapsed=True)


def show_conclusions(reviews,audit):
    observed=[r for r in reviews.values() if len(r['raw'])]
    connected=[r for r in reviews.values() if r['identity']['LOT_NO']]
    comparable=[r for r in reviews.values() if len(matching_rows(r,'REPAIR_START'))]
    display(HTML(f'<p><b>정비 {len(audit)}건 중 관측 연결 {len(observed)}건, 감지시각 LOT 연결 {len(connected)}건, '
                 f'착수 전·완료 후 동일 LOT 15분 비교 {len(comparable)}건.</b> 감지 연결은 '
                 +escape(' / '.join(f'{r["work"].WO_NO} ({r["work"].MNT_TYPE})' for r in connected))+'</p>'))
    rows=[]
    for r in observed:
        rows.append({'WO / 유형':f'{r["work"].WO_NO} / {r["work"].MNT_TYPE}','분석 질문':r['plan']['question'],'구체 관측':candidate_brief(block_candidate(r)),
            '해석':concise_conclusion(r),'완료 후':r['effect_verdict'],'추가 확인':r['plan']['missing']})
    show_table(pd.DataFrame(rows),'사례별 결론 · 운전 변화와 고장·조치 효과의 판단을 분리',height=500)
    display(HTML('<p><b>지금 확인한 것은 정비 전후의 관측 가능한 운전 양상이다.</b> '
        '계획점검 중 가열 4존의 공통 상승과 다른 정비의 중첩이 있어 온도 상승을 조치 효과로 해석할 수 없다. '
        '분석계 교체 사례는 감지 전후 CP 차이를 관측했지만 교체 후 측정이 없어 개선 여부를 검증하지 못했다. '
        '개별 마모·전기·트립 고장의 원인은 직접 신호와 부품 위치를 연결한 뒤 검증해야 한다.</p>'))

In [ ]:
def save_and_display(fig, filename):
    """Save the computed plot in the existing analysis folder, then display it inline."""
    if os.path.basename(filename) != filename or os.path.splitext(filename)[1].lower() != '.png':
        raise ValueError('저장 파일명은 PNG 이름만 지정하세요.')
    if not os.path.isdir(FIGURE_DIR):
        raise FileNotFoundError(f'기존 그림 폴더를 확인하세요: {FIGURE_DIR}')
    path=os.path.join(FIGURE_DIR, filename)
    fig.savefig(path,dpi=170,bbox_inches='tight',facecolor='white')
    SAVED_FIGURES[filename]=path
    display(fig)
    plt.close(fig)
    print(f'저장: {os.path.relpath(path, ROOT)}')
    return path


def style(ax):
    ax.spines[['top','right']].set_visible(False);ax.grid(alpha=.12);ax.tick_params(labelsize=9)


def date_axis(ax,lo,hi):
    ax.set_xlim(lo,hi);locator=mdates.AutoDateLocator(minticks=3,maxticks=5)
    ax.xaxis.set_major_locator(locator);ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    style(ax)


def observed_blocks(raw,merge_minutes=30):
    blocks=[]
    for _,g in raw.groupby('SEGMENT',sort=False):
        a,b=g.MEAS_DT.min(),g.MEAS_DT.max();lot,grade=g.LOT_NO.iloc[0],g.GRADE.iloc[0]
        if blocks and blocks[-1]['lot']==lot and blocks[-1]['grade']==grade and a-blocks[-1]['end']<=pd.Timedelta(minutes=merge_minutes):
            blocks[-1]['end']=b
        else:blocks.append(dict(start=a,end=b,lot=lot,grade=grade))
    return blocks


def channel_panel(fig,spec,title):
    inner=spec.subgridspec(3,1,height_ratios=[.30,.27,1.8],hspace=.04)
    title_ax=fig.add_subplot(inner[0]);title_ax.set_axis_off();title_ax.text(0,1,title,va='top',fontsize=10,color='#273e56')
    legend_ax=fig.add_subplot(inner[1]);legend_ax.set_axis_off()
    return fig.add_subplot(inner[2]),legend_ax


def plot_focus(ax,legend,raw,channels,focus=None):
    for i,key in enumerate(channels):
        color=COLORS[(int(key.split('Z')[-1])-3)%len(COLORS)] if key!='CP-Z4' else '#276a9a'
        selected=key==focus
        for j,(_,part) in enumerate(raw.groupby('SEGMENT',sort=False)):
            ax.plot(part.MEAS_DT,part[key],color=color,lw=1.15 if selected else .7,alpha=1 if selected else .55,
                    label=key if j==0 else None,zorder=4 if selected else 2)
    legend.legend(*ax.get_legend_handles_labels(),loc='center left',bbox_to_anchor=(0,.5),frameon=False,
                  borderaxespad=0,borderpad=0,ncol=len(channels),fontsize=8.5,handlelength=2,columnspacing=1.2)


def work_timeline(ax,data,work,raw):
    ax.set_axis_off();ax.set_xlim(0,1);ax.set_ylim(0,1)
    points=[(.03,'DETC_DT','감지 기록','D'),(.52,'STRT_DT','정비 시작','o'),(.97,'END_DT','정비 완료','s')]
    ax.annotate('',xy=(.515,.45),xytext=(.035,.45),arrowprops=dict(arrowstyle='->',color='#9b9b9b',linestyle=':',lw=1.8))
    ax.annotate('',xy=(.965,.45),xytext=(.525,.45),arrowprops=dict(arrowstyle='->',color=EVENT_COLORS['STRT_DT'],lw=3))
    ax.text(.275,.72,'감지 후~정비 시작 전 · '+elapsed_text(work.STRT_DT-work.DETC_DT),ha='center',fontsize=9,color='#6a665c')
    ax.text(.745,.72,'정비 작업 · '+elapsed_text(work.END_DT-work.STRT_DT),ha='center',fontsize=9,color='#6e4c83')
    for x,field,name,marker in points:
        stamp=getattr(work,field);ha='left' if x<.1 else 'right' if x>.9 else 'center'
        ax.scatter([x],[.45],s=45,marker=marker,color=EVENT_COLORS[field],zorder=5)
        ax.text(x,.09,f'{name} {stamp:%m-%d %H:%M}',ha=ha,va='top',fontsize=10,color=EVENT_COLORS[field])
    ax.text(1,1,'시간순서도 · 간격은 시간 길이에 비례하지 않음',ha='right',fontsize=7.5,color='#708090')


def block_relation(work,block):
    if block['end']<work.DETC_DT:return '감지 전 관측 · 감지 '+elapsed_text(work.DETC_DT-block['end'])+' 전 종료'
    if block['start']>=work.END_DT:return '완료 후 관측 · 완료 '+elapsed_text(block['start']-work.END_DT)+' 후 시작'
    if block['start']>=work.DETC_DT and block['end']<work.STRT_DT:
        return '감지 후~정비 시작 전 관측 · 측정 종료→정비 시작 '+elapsed_text(work.STRT_DT-block['end'])
    if block['start']<work.STRT_DT and block['end']>=work.END_DT:return '정비 시작 전·작업 중·완료 후 관측'
    if block['end']<work.STRT_DT:return '감지 기록 앞뒤 관측 · 정비 시작 전까지 측정'
    return '정비 시간과 겹친 관측'


def comparison_bundle(review):
    e=review['evidence']
    if e.empty:return None
    code='REPAIR_START' if repair_available(review) else 'DETC_DT'
    if code=='DETC_DT' and not review['identity']['LOT_NO']:return None
    rows=e.loc[e.CODE.eq(code)&e.WINDOW.eq(15)]
    if rows.empty:return None
    if review['plan']['primary']=='CP-Z4':groups=[(['CP-Z4'],'분위기 CP','CP[단위 미확인]')]
    else:
        groups=[(['HEAT_LEVEL']+TC[:4],'가열·균열존 온도','온도[℃]'),(['HEAT_OP']+OP[:4],'가열·균열존 출력','제어출력[%]')]
    return dict(code=code,rows=rows,groups=groups)


def comparison_panel(fig,spec,review,bundle):
    inner=spec.subgridspec(2,1,height_ratios=[1.1,2.0],hspace=.25)
    title=fig.add_subplot(inner[0]);title.set_axis_off()
    sample=bundle['rows'].loc[bundle['rows'].KEY.eq(review['plan']['primary'])].iloc[0]
    before_label='정비 시작 전 15분' if bundle['code']=='REPAIR_START' else '감지 기록 전 15분'
    after_label='정비 완료 후 15분' if bundle['code']=='REPAIR_START' else '감지 기록 후 15분'
    color_after='#23785b' if bundle['code']=='REPAIR_START' else '#c16d2a'
    title.text(0,1,f'같은 LOT {sample.LOT_NO} / {sample.GRADE} · {before_label} ↔ {after_label}',va='top',fontsize=11,weight='bold')
    times=f'전 {sample.BEFORE_START:%m-%d %H:%M}~{sample.BEFORE_END:%H:%M:%S} / 후 {sample.AFTER_START:%m-%d %H:%M}~{sample.AFTER_END:%H:%M:%S}'
    title.text(0,.5,times,va='top',fontsize=9,color='#52657a')
    title.legend(handles=[Line2D([],[],marker='o',ls='',color='#28689b',label=before_label),
                          Line2D([],[],marker='D',ls='',color=color_after,label=after_label)],
                 loc='upper right',frameon=False,ncol=2,fontsize=8.5,borderpad=0)
    columns=inner[1].subgridspec(1,len(bundle['groups']),wspace=.55)
    for i,(keys,label,ylabel) in enumerate(bundle['groups']):
        rows=bundle['rows'].set_index('KEY').reindex(keys).dropna(subset=['BEFORE','AFTER']).reset_index()
        if rows.empty:continue
        ax=fig.add_subplot(columns[0,i]);positions=np.arange(len(rows))
        for y,row in zip(positions,rows.itertuples()):
            ax.plot([row.BEFORE,row.AFTER],[y,y],color='#9daab5',lw=1.8,zorder=1)
            ax.scatter([row.BEFORE],[y],color='#28689b',s=32,zorder=3)
            ax.scatter([row.AFTER],[y],color=color_after,marker='D',s=32,zorder=4)
        labels=[]
        for row in rows.itertuples():
            name='4존 중앙' if row.KEY in EXTRA_FEATURES else 'CP-Z4' if row.KEY=='CP-Z4' else f'{int(row.KEY.split("Z")[-1])-2}존 {row.KEY}'
            unit='%p' if FEATURE_UNITS[row.KEY]=='%' else FEATURE_UNITS[row.KEY]
            labels.append(f'{name}  {value_text(row.KEY,row.DELTA,True)}{unit}')
        ax.set_yticks(positions,labels,fontsize=8.5);ax.invert_yaxis();ax.set_ylim(len(rows)-.5,-.5)
        ax.set_title(label+', 행별 숫자는 이후-이전 차이',fontsize=9,loc='left',pad=8)
        ax.set_xlabel(ylabel,fontsize=9);style(ax);ax.tick_params(axis='x',labelsize=8)
        if ylabel=='제어출력[%]':ax.set_xlim(0,100)


def sensor_figure(data,raw,title,work=None,review=None):
    review=review or work_review(data,work);blocks=observed_blocks(raw)
    if not blocks:return
    bundle=comparison_bundle(review);gas=review['plan']['primary']=='CP-Z4';rows_per_block=3 if gas else 2
    heights=[.72,.85]
    for block in blocks:heights += [.72]+[1.75]*rows_per_block
    if bundle:heights += [3.65 if not gas else 2.8, .35]
    heights += [.34]
    fig=plt.figure(figsize=(18,sum(heights)+.6))
    grid=fig.add_gridspec(len(heights),1,left=.085,right=.98,top=.98,bottom=.03,height_ratios=heights,hspace=.20)
    header=fig.add_subplot(grid[0]);header.set_axis_off()
    header.text(0,1,f'{work.WO_NO} · {work.MNT_TYPE} | 기록: {work.REMARK.strip() or "비고 없음"}',va='top',fontsize=17,weight='bold',color='#21394b')
    action=review['plan']['action'];action='수행 조치 상세 미기록' if action.startswith('부품교체 기록 없음') else action
    header.text(0,.36,'조치 기록: '+action,va='top',fontsize=9.5,color='#52657a')
    badge='정비 전후 비교 가능' if repair_available(review) else '완료 후 관측만 있음' if raw.MEAS_DT.ge(work.END_DT).any() else '정비 이후 비교 불가'
    header.text(1,.36,badge,ha='right',va='top',fontsize=10.5,color='#23785b' if repair_available(review) else '#9c642d',weight='bold')
    timeline=fig.add_subplot(grid[1]);work_timeline(timeline,data,work,raw)
    row_index=2
    for number,block in enumerate(blocks,1):
        part=raw.loc[raw.MEAS_DT.between(block['start'],block['end'])];event=block_candidate(review,block)
        if gas:groups=[(CP,'가열·균열존 | 분위기 CP','CP[단위 미확인]'),(TC[:4],'가열·균열존 | 1~4존','온도[℃]'),(OP[:4],'가열·균열존 | 1~4존 제어출력','제어출력[%]')]
        else:
            aging=event is not None and int(event.TC.split('Z')[-1])>=7
            groups=[(TC[4:6] if aging else TC[:4],'과시효대 | 1·2존' if aging else '가열·균열존 | 1~4존','온도[℃]'),
                    (OP[4:] if aging else OP[:4],'과시효대 | 1·2존 제어출력' if aging else '가열·균열존 | 1~4존 제어출력','제어출력[%]')]
        group_titles=grid[row_index].subgridspec(1,2,width_ratios=[1.2,1],wspace=.22);row_index+=1
        left_title=fig.add_subplot(group_titles[0]);left_title.set_axis_off()
        left_title.text(0,1,f'LOT {block["lot"]} / {block["grade"]} · 전체 관측 {block["start"]:%m-%d %H:%M}~{block["end"]:%m-%d %H:%M}',va='top',fontsize=10.5,weight='bold')
        left_title.text(0,.38,block_relation(work,block),va='top',fontsize=9,color='#52657a')
        right_title=fig.add_subplot(group_titles[1]);right_title.set_axis_off()
        if event is not None:
            right_title.text(0,1,f'{number}번 관측변화 확대 · {event.START:%m-%d %H:%M}~{event.END:%H:%M:%S}',va='top',fontsize=10.5,weight='bold',color='#9b652b')
            right_title.text(0,.38,candidate_brief(event)+' · 앞/뒤 5분 차이',va='top',fontsize=9,color='#52657a')
            zoom=part.loc[part.MEAS_DT.between(event.START,event.END)]
        else:
            right_title.text(0,1,'연속 10분 비교자료 부족',va='top',fontsize=10);zoom=part
        for k,(channels,label,ylabel) in enumerate(groups):
            columns=grid[row_index].subgridspec(1,2,width_ratios=[1.2,1],wspace=.22);row_index+=1
            focus=(event.OP if ylabel=='제어출력[%]' else event.TC) if event is not None else None
            reference=data.raw.loc[data.raw.MEAS_DT.between(raw.MEAS_DT.min(),raw.MEAS_DT.max()),channels]
            low,high=reference.min().min(),reference.max().max()
            margin=max((high-low)*.07,.0002 if ylabel=='CP[단위 미확인]' else .3)
            limits=(0,100) if ylabel=='제어출력[%]' else (low-margin,high+margin)
            for j,series in enumerate([part,zoom]):
                ax,legend=channel_panel(fig,columns[0,j],label)
                plot_focus(ax,legend,series,channels,focus)
                lo,hi=series.MEAS_DT.min(),series.MEAS_DT.max();pad=max(pd.Timedelta(seconds=5),(hi-lo)*.012)
                date_axis(ax,lo-pad,hi+pad);ax.set_ylim(limits);ax.set_ylabel(ylabel,fontsize=10)
                if event is not None:
                    if j==0:
                        ax.axvspan(event.START,event.END,color='#e6b24a',alpha=.20,lw=0)
                        ax.text(event.MIDPOINT,.96,str(number),transform=ax.get_xaxis_transform(),ha='center',va='top',fontsize=9,color='#8b5f19',weight='bold')
                    else:ax.axvline(event.MIDPOINT,color='#a88e5d',ls=':',lw=1)
                for field,name in POINTS.items():
                    point=getattr(work,field)
                    if lo<=point<=hi:ax.axvline(point,color=EVENT_COLORS[field],ls='--',lw=1)
                if bundle and j==0:
                    b=bundle['rows'].loc[bundle['rows'].KEY.eq(review['plan']['primary'])].iloc[0]
                    for a,z,color in [(b.BEFORE_START,b.BEFORE_END,'#28689b'),(b.AFTER_START,b.AFTER_END,'#23785b' if bundle['code']=='REPAIR_START' else '#c16d2a')]:
                        if z>=lo and a<=hi:ax.axvspan(max(a,lo),min(z,hi),color=color,alpha=.12,lw=0)
                ax.tick_params(labelbottom=k==len(groups)-1)
    if bundle:comparison_panel(fig,grid[row_index],review,bundle);row_index+=2
    footer=fig.add_subplot(grid[row_index]);footer.set_axis_off()
    footer.text(0,.8,concise_conclusion(review),va='top',fontsize=9.3,color='#80582d')
    save_and_display(fig,f'02_{work.WO_NO}_정비상세.png')

In [ ]:
def plot_coverage(data,reviews,audit):
    shown=audit.loc[audit.OBSERVED].reset_index(drop=True);n=len(shown)
    fig=plt.figure(figsize=(18,8.6))
    grid=fig.add_gridspec(1,3,left=.19,right=.98,bottom=.10,top=.83,width_ratios=[4.8,2.0,3.8],wspace=.10)
    near=fig.add_subplot(grid[0]);later=fig.add_subplot(grid[1],sharey=near);verdict=fig.add_subplot(grid[2],sharey=near)
    fig.suptitle('CAL01 · 관측을 정비와 어디까지 연결할 수 있나?',x=.025,y=.985,ha='left',fontsize=19,weight='bold')
    connected=sum(bool(r['identity']['LOT_NO']) for r in reviews.values());comparable=sum(repair_available(r) for r in reviews.values())
    fig.text(.025,.935,f'정비 {len(audit)}건 중 관측 연결 {n}건 · 감지시각 LOT 연결 {connected}건 · 동일 LOT 정비 전후 비교 {comparable}건',fontsize=11)
    handles=[Line2D([],[],lw=6,color='#287f8e',label='실제 관측 LOT'),Line2D([],[],ls=':',color='#a89e8a',label='감지~정비 시작 기록'),
             Line2D([],[],lw=5,color='#87619f',label='정비 작업'),Line2D([],[],ls='',marker='D',color=EVENT_COLORS['DETC_DT'],label='감지 기록'),
             Line2D([],[],ls='',marker='o',color=EVENT_COLORS['STRT_DT'],label='정비 시작'),Line2D([],[],ls='',marker='s',color=EVENT_COLORS['END_DT'],label='정비 완료')]
    fig.legend(handles=handles,loc='upper left',bbox_to_anchor=(.02,.91),frameon=False,ncol=6,fontsize=9)
    def hours(stamp,work):return (stamp-work.DETC_DT).total_seconds()/3600
    for i,item in shown.iterrows():
        r=reviews[item['WO']];w=r['work']
        for ax in [near,later]:
            ax.axhline(i,color='#e7edf2',lw=.7,zorder=-5)
            ax.plot([0,hours(w.STRT_DT,w)],[i+.13]*2,color='#a89e8a',ls=':',lw=1.5)
            ax.plot([hours(w.STRT_DT,w),hours(w.END_DT,w)],[i+.13]*2,color='#87619f',lw=5,solid_capstyle='butt')
            for field,marker in [('DETC_DT','D'),('STRT_DT','o'),('END_DT','s')]:
                ax.scatter([hours(getattr(w,field),w)],[i+.13],s=24,marker=marker,color=EVENT_COLORS[field],zorder=5)
            for b in observed_blocks(r['raw']):
                a,z=hours(b['start'],w),hours(b['end'],w)
                ax.plot([a,z],[i-.15]*2,color='#287f8e',lw=6,solid_capstyle='butt')
                center=(a+z)/2
                if (ax is near and -24<=center<=24) or (ax is later and center>24):
                    ax.text(center,i-.31,f'{b["lot"][-3:]}/{b["grade"]}',ha='center',va='bottom',fontsize=7.5,color='#176879',clip_on=True)
        title,detail=relation_summary(r)
        color='#23785b' if repair_available(r) else '#9b652b'
        if w.WO_NO=='WO-2024-00627':title='LOT 037→038 · 강종도 다름';detail='둘 다 정비 시작 전 · 완료 후 비교 불가'
        verdict.text(.02,i-.20,title,va='center',fontsize=9.5,color=color,weight='bold')
        verdict.text(.02,i+.10,detail,va='center',fontsize=8.8,color='#536777')
        candidate=block_candidate(r)
        if candidate is not None:verdict.text(.02,i+.36,'관측변화: '+short_zone(candidate.TC),va='center',fontsize=8,color='#6a7885')
    near.set_xlim(-24,24);later.set_xlim(24,240)
    near.set_xticks([-24,-12,0,12,24],['−24h','−12h','감지','+12h','+24h'])
    later.set_xticks([24,72,120,168,216],['+1일','+3일','+5일','+7일','+9일'])
    labels=[f'{row["WO"][-5:]} · {row["유형"]} · {str(row["기록된 현상"]).strip() or "비고 없음"}\n감지 {row["감지기록"]:%m-%d %H:%M}' for _,row in shown.iterrows()]
    near.set_yticks(np.arange(n),labels,fontsize=9);near.set_ylim(n-.5,-.6)
    near.axvline(0,color=EVENT_COLORS['DETC_DT'],ls='--',lw=1,alpha=.65)
    near.set_title('감지 주변 · 시간 단위',loc='left',fontsize=11,pad=15)
    later.set_title('하루 이후 · 일 단위',loc='left',fontsize=11,pad=15)
    verdict.set_title('이 자료로 알 수 있는 것',loc='left',fontsize=11,pad=15)
    for ax in [near,later]:style(ax);ax.tick_params(axis='y',length=0)
    later.tick_params(labelleft=False);verdict.set_axis_off()
    fig.text(.025,.025,'감지를 0으로 맞춘 실제 시간관계 · 오른쪽은 축척 변경 · LOT는 240xxx / WO는 WO-2024-xxxxx · 기록 구간은 실제 정지구간과 다름',fontsize=9,color='#52657a')
    save_and_display(fig,'00_CAL01_정비사유별_분석범위.png')


def plot_year(data,year=2024):
    labels=['BM 사후정비','CBM 상태기반정비','TBM 시간기준정비','부품교체 연결','실측 LOT','소둔로 수리계획']
    pos={name:5-i for i,name in enumerate(labels)};type_y={'BM':5,'CBM':4,'TBM':3}
    observed={wo for wo,r in reviews.items() if len(r['raw'])}
    fig,axes=plt.subplots(4,1,figsize=(18,12.0),sharey=True)
    fig.subplots_adjust(left=.13,right=.98,top=.88,bottom=.055,hspace=.52)
    fig.suptitle(f'{year} · CAL01 정비·교체·관측 날짜',x=.025,y=.985,ha='left',fontsize=19,weight='bold')
    fig.text(.025,.945,f'정비 {len(data.maintenance)}건 · 교체기록 {data.parts.WO_NO.nunique()} WO · 관측 {len(data.lots)} LOT | 모든 분기에서 같은 행과 순서 사용',fontsize=11)
    for q,ax in enumerate(axes):
        lo=pd.Timestamp(year=year,month=q*3+1,day=1);hi=lo+pd.DateOffset(months=3)
        tasks=data.maintenance.loc[data.maintenance.DETC_DT.lt(hi)&data.maintenance.END_DT.ge(lo)]
        for w in tasks.itertuples():
            y=type_y[w.MNT_TYPE];color=TYPE_COLORS[w.MNT_TYPE]
            ax.plot([w.DETC_DT,w.STRT_DT],[y,y],color=color,ls=':',lw=1.3)
            ax.plot([w.STRT_DT,w.END_DT],[y,y],color=color,lw=6,solid_capstyle='butt')
            for field,marker in [('DETC_DT','D'),('STRT_DT','o'),('END_DT','s')]:
                stamp=getattr(w,field)
                if lo<=stamp<hi:ax.scatter([stamp],[y],s=20,marker=marker,color=color,zorder=4)
            if w.WO_NO in observed and lo<=w.DETC_DT<hi:
                ax.annotate(w.WO_NO[-5:],(w.DETC_DT,y),xytext=(0,9),textcoords='offset points',ha='center',fontsize=7.5,color=color)
            if w.WO_NO in set(data.parts.WO_NO):
                ax.barh([pos['부품교체 연결']],[(w.END_DT-w.STRT_DT).total_seconds()/86400],left=mdates.date2num(w.STRT_DT),height=.38,
                        facecolor='#e9ddef',edgecolor='#87619f',hatch='///')
        obs=data.raw.loc[data.raw.MEAS_DT.ge(lo)&data.raw.MEAS_DT.lt(hi)]
        for lot,g in obs.groupby('LOT_NO',sort=False):
            for _,part in g.groupby('SEGMENT',sort=False):
                ax.plot([part.MEAS_DT.min(),part.MEAS_DT.max()],[pos['실측 LOT']]*2,color='#287f8e',lw=6,solid_capstyle='butt')
            ax.annotate(lot[-3:],(g.MEAS_DT.min(),pos['실측 LOT']),xytext=(0,8),textcoords='offset points',ha='center',fontsize=7.5,color='#176879')
        for plan in data.plans.loc[data.plans.START.lt(hi)&data.plans.END_EXCL.gt(lo)].itertuples():
            ax.plot([plan.START,plan.END_EXCL],[pos['소둔로 수리계획']]*2,color='#899b37',lw=7,solid_capstyle='butt')
        ax.set_yticks(list(pos.values()),labels,fontsize=9);ax.set_ylim(-.65,5.65);ax.set_xlim(lo,hi);style(ax)
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO,interval=2));ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
        ax.set_title(f'{q*3+1}~{q*3+3}월',loc='left',fontsize=11,pad=12)
    fig.text(.025,.015,'◆ 감지 · ● 정비 시작 · ■ 완료 · 점선 감지~시작 · 굵은선 작업 | 번호는 상세 관측이 있는 WO 끝 5자리 / LOT 240xxx',fontsize=9,color='#52657a')
    save_and_display(fig,f'01_CAL01_전체시간흐름_{year}.png')


def show_work(data,wo_no,review=None):
    matches=data.maintenance.loc[data.maintenance.WO_NO.eq(wo_no)]
    if len(matches)!=1:raise ValueError(f'CAL01 WO 번호를 확인하세요: {wo_no}')
    work=next(matches.itertuples());review=review or work_review(data,work)
    if review['raw'].empty:return review
    sensor_figure(data,review['raw'],wo_no,work=work,review=review)
    show_evidence(review)
    show_history(history_window(data,work_windows(work)[0][1],work_windows(work)[-1][2]),'선택 작업 전후의 기록된 사건·조치 · 동시 작업 포함')
    return review

In [ ]:
furnace = load_furnace_data()
display(HTML('<style>.furnace-table{border-collapse:collapse;font-size:12px}.furnace-table th{position:sticky;top:0;background:#e9f0f6;padding:8px;text-align:left}.furnace-table td{padding:8px;border-bottom:1px solid #dce4ec;vertical-align:top;min-width:80px}</style>'))


## 검토 기준

| 정비 기록 | 우선 검토할 신호와 질문 | 직접 확인에 필요한 자료 |
|---|---|---|
| TBM 계획점검 | 가열 4존 공통 변화·존간 차이·출력·CP의 전후 차이 | 목표값·운전 상태. 기록시각을 고장 발생으로 해석하지 않음 |
| 전기·동작·트립 | 같은 존의 온도와 제어출력, 원본 출력 0·100 비중 | 전류·절연값·회전수·트립/운전 상태·부품 설치 위치 |
| 가스 분석계 셀 교체 | 분위기 CP의 수준·변동 폭; 온도·출력은 운전 맥락 | 분석계 원지시값·교정 결과·설치 위치. CP는 간접지표 |
| 마모·현상 미특정 | 운전 변화의 존재만 검토 | 마모량·대상 부품·위치. 온도만으로 마모를 진단하지 않음 |

- 정비유형·비고·부품명으로 검토 질문을 정한 분석상 분류다. 실제 고장 부품과 센서의 연결표는 제공되지 않았다. 원문 충돌 가능성은 별도 표시한다.
- 조회 범위는 감지 전 24시간, 대기·작업 전체, 완료 후 24시간이다. 센서가 없는 기간도 정비 기록은 보존한다.
- 원본을 보존하고 LOT·연결 강종·연속구간별로 분리한다. 유효 1분창은 채널별 30개 이상 측정, 측정 시각 폭 45초 이상이다. 관측 공백과 서로 다른 LOT·강종을 이어 붙이지 않는다.
- 감지/착수/완료 경계와 착수 전→완료 후를 **5·15·30분**에서 비교한다. 15분은 설명용 기본 창이며 물리적 응답시간으로 검증된 기준이 아니다. 각 창에 양쪽 3분창 이상 있어야 수치를 내고, 양쪽 80% 이상 관측된 창만 민감도 요약에 쓴다. 관측 비율은 분석 품질 기준이다.
- 중앙값(수준), IQR(분창 사이 변동 폭), 연속 1분 변화 절댓값의 중앙값(국소 변화), 원본 출력 0·100 비중을 함께 본다. 승온 중 IQR 증가를 제어 불안정으로 단정하지 않는다. 존간 최대-최소도 서로 다른 목표값 때문에 생길 수 있다.
- 짧은 출력 단절이 중앙값에 가려지지 않도록, 원본 출력이 정확히 0인 연속 1초 표본 2개 이상의 구간도 따로 찾는다. 1초 공백·LOT/강종·정비 단계가 바뀌면 끊고, 단계·존별 최장 구간과 동시 온도·CP 범위를 표시한다. 붉은 띠는 관측 구간 표시이며 알람 기준이 아니다.
- 온도·출력은 제어 목표와 함께 해석해야 한다. [Yokogawa의 온도제어 설명](https://www.yokogawa.com/us/library/resources/media-publications/how-to-select-the-right-temperature-controller-for-your-application/)은 측정값·목표값·출력의 관계를 설명한다. 여기에는 목표값·라인 속도·운전 모드가 없어 목표 이탈·정상화·효율 개선을 판정하지 않는다.
- 자료는 교육용 재현 데이터이며, 존 명칭·단위·강종 연결은 로컬 데이터사전의 표시 기준이다. 고장모드 배정과 실제 열 신호 사이의 인과관계가 검증된 자료가 아니다. 공통 승온·반복 변동은 관측 사실로 기록하고 원인은 후보로만 해석한다.
- **관측변화 확대창:** 같은 LOT·강종·연속구간의 10개 연속 유효 1분창을 앞 5분/뒤 5분으로 나누고, 6개 존의 온도 변화와 같은 존 출력 변화를 함께 살핀다. `|온도 중앙값 차이| × |출력 중앙값 차이|`가 큰 창을 각 관측 블록의 표시 후보로 고른다. 분석계 교체는 CP 차이로 고른다. 고장 임계값·이상 점수·실제 감지 원인을 뜻하지 않으며, 모든 존 후보는 접힌 표에서 확인한다.
- 가능한 경우 BM·CBM은 감지 전 관측에서, TBM은 정비 시작 전 관측에서 확대창을 찾는다. 해당 연속 자료가 없으면 그 블록의 실제 관측에서만 고른다. 감지 뒤나 완료 뒤에 측정된 후보를 감지 전 징후로 바꾸어 쓰지 않는다.
- 상세 그림 왼쪽은 관측 전체, 오른쪽은 번호를 붙인 동일 구간의 확대다. 두 그림은 같은 y축 범위를 사용한다. 출력은 0~100%로 고정한다. 관측 전체의 노란 구간과 오른쪽 확대가 대응하며, 가운데 점선은 앞/뒤 5분을 나누는 시각이다.
- 정비 전후 비교가 가능한 경우 같은 LOT·강종의 정비 시작 전/완료 후 각 15분을 파랑/초록 구간과 연결점으로 표시한다. CP 사례의 감지 앞뒤 비교는 파랑/주황으로 표시해 정비 완료 후 비교와 구분한다. 점의 위치와 행별 차이값으로 수준 변화가 어느 존에서 관측됐는지 읽는다.
- 시간순서도는 감지→정비 시작→완료와 실제 소요시간을 표시하며 노드 간격은 시간 길이에 비례하지 않는다. 관측이 감지/완료에서 얼마나 떨어져 있는지는 각 LOT 제목과 전체 관계 그림에 표시한다. 연간 그림의 6개 행은 모든 분기에서 고정한다.

![CAL01 정비 사유별 분석 범위](<../../figures/jinny/06_maintenance_timeline_jinny/00_CAL01_정비사유별_분석범위.png>)

## 1. 관측과 정비의 시간관계

감지 기록을 0으로 맞추고 실제 관측 LOT, 정비 시작·완료를 같은 행에서 연결한다. 오른쪽에는 감지 연결·정비 이후 비교 가능 여부를 표시한다.
연간 그림은 모든 분기에서 BM·CBM·TBM·교체·실측 LOT·소둔로 수리계획의 6개 행을 고정한다.

In [ ]:
YEAR = 2024
START, END = None, None
reviews, audit = build_reviews(furnace)
plot_coverage(furnace, reviews, audit)
show_audit(audit)
plot_year(furnace, YEAR)

![11월 정비 WO-2024-01859 관측과 비교](<../../figures/jinny/06_maintenance_timeline_jinny/02_WO-2024-01859_정비상세.png>)

## 2. 무엇이 어느 존에서 변했고, 정비 후에도 비교할 수 있는가

각 LOT의 전체 관측과 번호를 붙인 변화 구간 확대를 나란히 본다. 온도와 같은 존의 출력을 함께 읽으며, 모든 존의 검토 수치는 접힌 표에 남긴다.
정비 전후 자료가 있으면 같은 LOT·강종의 각 15분 연결점 그림을 추가한다. 자료가 없으면 빈 비교 그래프를 만들지 않는다. 그림 아래 해석은 한 줄로 제한한다.

In [ ]:
observed_work = audit.loc[audit.OBSERVED, "WO"].tolist()
for wo_no in observed_work:
    show_work(furnace, wo_no, review=reviews[wo_no])

## 3. 무엇을 알았고, 무엇이 더 필요한가

수치 차이와 원인·정상화 판단을 분리한다. 아래 결론은 위 분석 결과로 생성되며, 관측이 없는 단계를 보완해서 만들지 않는다.

In [ ]:
show_conclusions(reviews, audit)

### 자료 해석과 후속 확인

실제 정지 시작·끝, 알람 태그·판정값, 부품 설치 존, 목표 온도·라인 속도·운전 모드가 없다. DOWN_MIN은 정지 길이이며 작업구간을 정지구간으로 대체하지 않는다.
CR1 과거 강종 변경 기록을 측정시각에 연결한 값은 소재별 확정 강종이 아니다. 감지시점 측정이 없으면 LOT를 배정하지 않는다.
다음 분석을 위해 WO와 알람 ID·설치 존을 연결하고, RUN/STOP·목표 온도·속도·직접 고장 신호·교체/교정 시각을 확보해야 한다.